# Clase 7. Aproximación de funciones y Deep Q-Networks (DQN)

**DIH DATalife — Curso de Reinforcement Learning (RL) desde cero**

Piensa en un tablero de ajedrez: cada disposición posible de las piezas es un estado distinto, y el número de estados totales supera con creces la cantidad de átomos del universo observable. O piensa en un videojuego, donde cada fotograma de píxeles en pantalla es, en la práctica, un estado nuevo. Hasta la clase seis guardabas un valor exacto por cada par estado-acción en una tabla. Hoy esa tabla deja de ser viable, y la sustituyes por una red neuronal que generaliza.

## Siglas de esta clase

- **RL**: Aprendizaje por Refuerzo (Reinforcement Learning)
- **DQN**: Deep Q-Network, red Q profunda
- **TD**: Diferencia Temporal (Temporal Difference)

## Guion de la clase

1. De tablas a funciones: la idea de la aproximación de funciones
2. Por qué unir Q-learning y una red neuronal, sin más, es inestable
3. Las dos piezas que estabilizan DQN: buffer de repetición y red objetivo
4. Código en vivo: DQN completo sobre CartPole, con PyTorch y gymnasium
5. Puntos clave y qué te espera en la clase ocho


## 1. De tablas a funciones: la aproximación de funciones

*Guion de clase: retoma la analogía del ajedrez de la portada. Pregunta a la clase cuántas veces creen que un agente de Q-learning tabular visitaría dos veces exactamente el mismo estado en un videojuego real. La respuesta —casi nunca— es la puerta de entrada a todo el tema de hoy.*

La tabla de Q-learning tabular funciona de maravilla en problemas pequeños, pero se rompe en cuanto el número de estados se dispara: no cabe en memoria y, sobre todo, tu agente casi nunca vuelve a visitar exactamente el mismo estado dos veces. Sin una tabla que lo recuerde, no aprende nada útil sobre un estado que ve una sola vez.

En RL, un agente aprende siempre alguna de estas tres cosas: una política (qué acción tomar en cada estado), una función de valor (cuánto vale cada estado o cada par estado-acción) o un modelo (a qué estado te lleva cada acción). Las tres son, en el fondo, funciones. La **aproximación de funciones** consiste en representar esa función con un conjunto de parámetros ajustables, en vez de con una tabla exacta:

`Q(s,a;θ) ≈ Q_verdadero(s,a)`

donde θ representa esos parámetros. El objetivo del entrenamiento es encontrar los valores de θ que acercan esa aproximación al valor real todo lo posible.

Una aclaración breve, por si las redes neuronales todavía te resultan un territorio nuevo: una red neuronal es, en el fondo, una función con parámetros ajustables (los pesos), organizada en capas que transforman la entrada paso a paso hasta producir una salida. Entrenarla consiste en mover esos pesos, poco a poco y de forma automática, hasta que la función que calcula se parezca cada vez más a la función que quieres imitar; en tu caso, el valor de cada acción posible en cada estado.

**¿Por qué merece la pena el cambio?** Por dos razones:

- **Generalización**: un agente que aproxima Q con una función puede estimar el valor de un estado que nunca visitó, apoyándose en estados parecidos que sí conoce
- **Menos exigencia por paso**: ya no necesitas optimizar la política a la perfección en cada ciclo; basta con mejorar la aproximación de forma incremental para que la política mejore con ella

Existen varios tipos de aproximadores —combinaciones lineales de características hechas a mano, árboles de decisión, vecinos más próximos— pero hoy dominan las redes neuronales, porque aprenden directamente de señales en bruto, como los píxeles de una pantalla, sin que tengas que diseñar ninguna característica a mano. La idea no es nueva: en los años ochenta, Gerald Tesauro combinó aprendizaje por Diferencia Temporal (TD) con una red neuronal para jugar al backgammon, y Chris Watkins formuló Q-learning en 1989. Lo que ha cambiado desde entonces es la capacidad de cómputo disponible.


## 2. Por qué unir Q-learning y una red neuronal, sin más, es inestable

*Guion de clase: plantea la versión ingenua en la pizarra antes de explicar por qué falla. Deja que la clase intuya el problema antes de darlo.*

La versión ingenua es tentadora: sustituye la tabla de Q-learning por una red neuronal que reciba el estado y devuelva un valor por cada acción, elige la acción con épsilon-greedy, calcula el error entre lo que predijo la red y lo que "debería" haber predicho, y ajusta los pesos con descenso de gradiente. Sobre el papel funciona. En la práctica, entrenada así, la red suele oscilar sin converger o directamente diverge, por dos motivos.

**Primer motivo: los datos no son independientes.** Los pasos consecutivos de un episodio están muy correlacionados entre sí: el estado en el paso cien se parece muchísimo al estado en el paso 101. Es la diferencia entre entrenar con fotos al azar de miles de escenas distintas y entrenar viendo, fotograma a fotograma, los mismos segundos de una única película: la red se especializa en lo que acaba de ver y olvida lo que vio antes.

**Segundo motivo: el bootstrapping mueve el propio objetivo.** En Q-learning, el objetivo (target) que persigue tu red se calcula usando la propia red:

`objetivo = r + γ · max_a' Q(s',a';θ)`

Cada vez que ajustas los pesos θ para acercarte a ese objetivo, el objetivo también se mueve, porque depende de esos mismos pesos. Es como intentar dar en una diana que se desplaza cada vez que disparas, y que además se desplaza justamente porque disparaste. A eso súmale que la política que decide qué datos recoge el agente también cambia con la red, así que los datos de entrenamiento van cambiando a la vez que la propia red que intentas entrenar.


## 3. Las dos piezas que estabilizan DQN

*Guion de clase: presenta cada pieza como la respuesta directa a uno de los dos motivos de inestabilidad del bloque anterior. El buffer ataca la correlación entre pasos; la red objetivo ataca el bootstrapping.*

DQN (Mnih y colaboradores, 2015) resuelve ambos problemas añadiendo dos ingredientes al Q-learning con redes neuronales.

**Buffer de repetición de experiencias (experience replay).** En vez de entrenar con la transición que el agente acaba de vivir y descartarla, guarda cada transición `(s, a, r, s')` en una memoria y, en cada paso de entrenamiento, muestrea al azar un mini-lote de esa memoria. Esto rompe la correlación entre pasos consecutivos, reutiliza cada transición varias veces y suaviza la distribución de los datos de entrenamiento, porque combina experiencia recogida bajo políticas ligeramente distintas a lo largo del entrenamiento.

**Red objetivo (target network).** Mantén una segunda copia de la red, con sus propios parámetros θ⁻, y utilízala solo para calcular el objetivo de entrenamiento. Esta red no se actualiza en cada paso: se sincroniza con la red principal cada cierto número fijo de pasos, copiando sus pesos, y permanece congelada mientras tanto. Así, el objetivo deja de moverse en cada paso y se convierte en una referencia estable durante un tramo de entrenamiento.

Con ambas piezas, la función de pérdida de DQN queda así, de forma intuitiva:

`L(θ) = E[ (r + γ · max_a' Q(s',a';θ⁻) − Q(s,a;θ))² ]`

donde la esperanza se calcula sobre mini-lotes muestreados del buffer, θ son los parámetros de la red principal (la que se entrena) y θ⁻ los de la red objetivo (congelada entre sincronizaciones). Un detalle técnico importante: al calcular el gradiente, aplicas un "stop-gradient" sobre el término del objetivo, de modo que el ajuste de pesos solo fluye hacia θ, nunca hacia θ⁻. Sin ese detalle, estarías moviendo el propio objetivo para que se acerque a tu predicción, en vez de al revés.

La red objetivo sigue siendo un componente estándar en la inmensa mayoría de implementaciones de DQN, junto con un factor de descuento típico cercano a 0,99 y una exploración épsilon-greedy con decaimiento (vas a poner esto último a prueba en la práctica de hoy).

**Tres mejoras posteriores**, que vas a implementar en la clase diez:

- **Double DQN** separa la elección de la mejor acción siguiente (con la red principal) de su evaluación (con la red objetivo), para reducir el sesgo de sobreestimación típico de Q-learning
- **Dueling DQN** divide la red en dos ramas —una que estima el valor del estado y otra que estima la ventaja de cada acción— y las combina al final
- **Prioritized Experience Replay** muestrea el buffer dando más peso a las transiciones con mayor error de predicción, en vez de muestrear todas por igual


## 4. Código en vivo: DQN sobre CartPole

*Guion de clase: construye el ejemplo celda a celda, sin adelantar código. Antes de ejecutar el bucle de entrenamiento, pide a la clase que prediga cómo será la curva de recompensa.*

Vas a entrenar un DQN sobre CartPole, el entorno clásico en el que un carrito debe mantener un péndulo en equilibrio moviéndose a izquierda o derecha. El estado son cuatro números (posición y velocidad del carrito, ángulo y velocidad angular del péndulo) y hay dos acciones posibles. Es mucho más simple que Atari, pero contiene exactamente los mismos ingredientes: red Q, buffer de repetición y red objetivo.

**Nota de diseño para hoy:** para que puedas seguir el código en vivo sin distracciones, esta clase usa un epsilon **fijo** durante todo el entrenamiento (no decae). En la práctica vas a comparar esta versión con una que sí incorpora decaimiento de epsilon, tal como recomienda el manual.

La única dependencia externa de esta clase es `gymnasium`, la librería que provee el entorno CartPole. PyTorch, `numpy` y `matplotlib` ya vienen preinstalados en Google Colab.


In [ ]:
# Instalamos gymnasium: es la UNICA dependencia externa de esta clase.
# PyTorch, numpy y matplotlib ya vienen preinstalados en Google Colab.
!pip install gymnasium


### Importaciones y semilla

Fijamos una semilla para que el experimento sea, en la medida de lo posible, reproducible.


In [ ]:
import random
from collections import deque, namedtuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import gymnasium as gym

SEMILLA = 42
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)


### Hiperparámetros

Ajustados para entrenar rápido en una CPU normal de Colab, priorizando claridad sobre rendimiento máximo. `EPSILON` queda fijo durante todo el entrenamiento: es la simplificación de esta clase que vas a poner en cuestión en la práctica.


In [ ]:
NOMBRE_ENTORNO = "CartPole-v1"
NUM_EPISODIOS = 200                    # episodios de entrenamiento
GAMMA = 0.99                           # factor de descuento
TASA_APRENDIZAJE = 1e-3
TAMANO_BUFFER = 10000                  # capacidad del buffer de repeticion
TAMANO_LOTE = 64                       # tamano del mini-lote muestreado en cada paso
EPSILON = 0.1                          # exploracion epsilon-greedy, FIJA en esta clase
PASOS_ACTUALIZAR_RED_OBJETIVO = 500    # cada cuantos pasos se sincroniza la red objetivo
PASOS_MINIMOS_ANTES_ENTRENAR = 1000    # esperamos a tener algo de experiencia antes de entrenar


### La red Q: el aproximador de funciones

Recibe un estado (cuatro números, en CartPole) y devuelve un valor estimado para cada acción posible. Es un perceptrón multicapa pequeño, suficiente para un estado tan corto.


In [ ]:
class RedQ(nn.Module):
    """Aproximador de funciones: recibe un estado y devuelve un valor
    estimado para cada accion posible."""

    def __init__(self, dim_estado, num_acciones):
        super().__init__()
        self.capas = nn.Sequential(
            nn.Linear(dim_estado, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, num_acciones),
        )

    def forward(self, estado):
        return self.capas(estado)


### El buffer de repetición de experiencias

Guarda transiciones pasadas y permite muestrear mini-lotes al azar en vez de entrenar solo con la transición que se acaba de vivir. `Transicion` es una tupla con nombre, para que el código sea legible al construir los mini-lotes.


In [ ]:
Transicion = namedtuple(
    "Transicion", ["estado", "accion", "recompensa", "siguiente_estado", "terminado"]
)


class BufferRepeticion:
    """Rompe la correlacion entre pasos consecutivos y hace que los datos de
    entrenamiento se parezcan mas a un conjunto i.i.d. (ver bloque 2 de esta clase)."""

    def __init__(self, capacidad):
        # deque con tamano maximo: al llenarse, descarta la transicion mas
        # antigua para dejar sitio a la nueva (comportamiento tipo FIFO)
        self.memoria = deque(maxlen=capacidad)

    def guardar(self, *args):
        self.memoria.append(Transicion(*args))

    def muestrear(self, tamano_lote):
        return random.sample(self.memoria, tamano_lote)

    def __len__(self):
        return len(self.memoria)


### Política épsilon-greedy

Con probabilidad epsilon, elige una acción al azar (exploración); si no, elige la acción con mayor valor Q según la red online (explotación). Es la misma política que ya conoces de clases anteriores.


In [ ]:
def elegir_accion(red_online, estado, epsilon, num_acciones):
    """Politica epsilon-greedy sobre los valores Q que estima la red online."""
    if random.random() < epsilon:
        return random.randrange(num_acciones)
    with torch.no_grad():
        estado_tensor = torch.tensor(estado, dtype=torch.float32).unsqueeze(0)
        valores_q = red_online(estado_tensor)
        return int(torch.argmax(valores_q, dim=1).item())


### El paso de entrenamiento: aquí vive la pérdida de DQN

*Guion de clase: para aquí y señala línea a línea la correspondencia con la fórmula del bloque tres: `Q(s,a;θ)` es `valores_q`, `θ⁻` es `red_objetivo`, y `torch.no_grad()` es el "stop-gradient".*

Muestrea un mini-lote del buffer y ajusta la red online para acercar su predicción al objetivo, calculado con la red objetivo (target network).


In [ ]:
def entrenar_paso(red_online, red_objetivo, buffer, optimizador):
    """Un paso de entrenamiento de DQN: muestrea un mini-lote del buffer y
    ajusta la red online para acercar su prediccion al objetivo calculado
    con la red objetivo (target network)."""
    lote = buffer.muestrear(TAMANO_LOTE)
    estados = torch.tensor(np.array([t.estado for t in lote]), dtype=torch.float32)
    acciones = torch.tensor([t.accion for t in lote], dtype=torch.int64).unsqueeze(1)
    recompensas = torch.tensor([t.recompensa for t in lote], dtype=torch.float32)
    siguientes_estados = torch.tensor(
        np.array([t.siguiente_estado for t in lote]), dtype=torch.float32
    )
    terminados = torch.tensor([t.terminado for t in lote], dtype=torch.float32)

    # Q(s,a;theta): valor que la red online asigna a la accion que de verdad se tomo
    valores_q = red_online(estados).gather(1, acciones).squeeze(1)

    # Objetivo: r + gamma * max_a' Q(s',a';theta-), calculado con la red objetivo.
    # torch.no_grad() implementa el "stop-gradient": el error se propaga
    # solo hacia la red online, nunca hacia la red objetivo.
    with torch.no_grad():
        max_q_siguiente = red_objetivo(siguientes_estados).max(dim=1)[0]
        objetivo = recompensas + GAMMA * max_q_siguiente * (1 - terminados)

    perdida = nn.functional.mse_loss(valores_q, objetivo)

    optimizador.zero_grad()
    perdida.backward()
    optimizador.step()


### Preparamos entorno, redes, buffer y optimizador

Creamos dos copias de `RedQ`: `red_online` (la que se entrena, θ) y `red_objetivo` (se congela entre sincronizaciones, θ⁻). Al empezar, ambas comparten los mismos pesos.


In [ ]:
entorno = gym.make(NOMBRE_ENTORNO)
dim_estado = entorno.observation_space.shape[0]
num_acciones = entorno.action_space.n

red_online = RedQ(dim_estado, num_acciones)     # red principal: la que se entrena (theta)
red_objetivo = RedQ(dim_estado, num_acciones)   # red objetivo: congelada entre sincronizaciones (theta-)
red_objetivo.load_state_dict(red_online.state_dict())  # al empezar, theta- = theta
red_objetivo.eval()

optimizador = optim.Adam(red_online.parameters(), lr=TASA_APRENDIZAJE)
buffer = BufferRepeticion(TAMANO_BUFFER)

print("Dimension del estado:", dim_estado, "| Numero de acciones:", num_acciones)


### El bucle de entrenamiento

**Ejecuta la celda siguiente.** Tarda solo unos segundos en una CPU normal. Fíjate en tres cosas mientras corre:

- Solo entrenamos cuando el buffer ya acumuló suficiente experiencia (`PASOS_MINIMOS_ANTES_ENTRENAR`)
- La red objetivo se sincroniza cada `PASOS_ACTUALIZAR_RED_OBJETIVO` pasos, no en cada paso
- `EPSILON` se mantiene fijo en 0,1 durante toda la corrida: en la práctica de hoy vas a cambiar esto


In [ ]:
recompensas_por_episodio = []
pasos_totales = 0

for episodio in range(NUM_EPISODIOS):
    estado, _ = entorno.reset(seed=SEMILLA + episodio)
    recompensa_episodio = 0.0
    terminado = False
    truncado = False

    while not (terminado or truncado):
        accion = elegir_accion(red_online, estado, EPSILON, num_acciones)
        siguiente_estado, recompensa, terminado, truncado, _ = entorno.step(accion)

        buffer.guardar(estado, accion, recompensa, siguiente_estado, terminado)
        estado = siguiente_estado
        recompensa_episodio += recompensa
        pasos_totales += 1

        # Solo entrenamos cuando el buffer ya acumulo suficiente experiencia
        if len(buffer) >= PASOS_MINIMOS_ANTES_ENTRENAR:
            entrenar_paso(red_online, red_objetivo, buffer, optimizador)

        # Sincronizamos la red objetivo cada cierto numero de pasos, no en cada paso
        if pasos_totales % PASOS_ACTUALIZAR_RED_OBJETIVO == 0:
            red_objetivo.load_state_dict(red_online.state_dict())

    recompensas_por_episodio.append(recompensa_episodio)

    if (episodio + 1) % 20 == 0:
        media_reciente = np.mean(recompensas_por_episodio[-20:])
        print(
            f"Episodio {episodio + 1}/{NUM_EPISODIOS} | epsilon={EPSILON:.2f} "
            f"| recompensa media (ultimos 20)={media_reciente:.1f}"
        )

entorno.close()


### Graficamos la recompensa por episodio

**Ejecuta esta celda para ver la curva completa.**


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(recompensas_por_episodio, alpha=0.4, label="Recompensa por episodio")
# Media movil de 20 episodios, para ver la tendencia sin el ruido episodio a episodio
media_movil = np.convolve(recompensas_por_episodio, np.ones(20) / 20, mode="valid")
plt.plot(range(19, len(recompensas_por_episodio)), media_movil, label="Media movil (20 episodios)")
plt.xlabel("Episodio")
plt.ylabel("Recompensa total")
plt.title("DQN en CartPole-v1: recompensa por episodio (epsilon fijo = 0.1)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


**Lectura del resultado**

*Guion de clase: para en esta gráfica y pregunta a la clase qué tramo de la curva corresponde a "la red todavía no ha aprendido nada".*

Al principio, la recompensa por episodio es baja e irregular: el péndulo se cae casi de inmediato porque la red apenas ha visto experiencia. A medida que el buffer se llena y la red entrena, la media móvil debería subir, aunque con oscilaciones: esa mezcla de progreso y ruido episodio a episodio es normal en DQN, incluso con buffer de repetición y red objetivo funcionando bien.


## Puntos clave

- Los métodos tabulares fallan cuando el número de estados crece demasiado: no caben en memoria y el agente rara vez repite el mismo estado
- La aproximación de funciones sustituye la tabla por una función parametrizada, `Q(s,a;θ)`, capaz de generalizar a estados nuevos
- Una red neuronal es, en esencia, una función con pesos ajustables que se entrena para parecerse cada vez más a la función que quieres aproximar
- Unir Q-learning con una red neuronal de forma ingenua es inestable, porque los datos de RL no son i.i.d. y el objetivo de entrenamiento se mueve a la vez que la red que lo persigue (bootstrapping)
- El buffer de repetición de experiencias rompe la correlación entre pasos consecutivos y reutiliza cada transición varias veces
- La red objetivo congela una copia de los parámetros durante un tramo de entrenamiento, así el objetivo deja de moverse en cada paso y el entrenamiento se estabiliza
- Double DQN, Dueling DQN y Prioritized Experience Replay son mejoras sobre DQN que vas a implementar en la clase diez

## Ponlo en práctica

Toma el código de esta clase y modifícalo tú misma o tú mismo:

- **Cambia `PASOS_ACTUALIZAR_RED_OBJETIVO`** (prueba con 50, con 500 y con 5.000) y observa qué pasa con la estabilidad de la curva: una red objetivo que se sincroniza demasiado rápido se parece cada vez más al Q-learning ingenuo e inestable del bloque dos
- **Cambia `TAMANO_BUFFER`** (prueba con 500 y con 50.000) y comprueba si un buffer pequeño vuelve el entrenamiento más errático, al perder diversidad de experiencias pasadas
- **Elimina temporalmente la red objetivo** (usa `red_online` en vez de `red_objetivo` dentro de `entrenar_paso`) y compara la curva resultante con la versión original

## Antes de la próxima clase

**Abre el notebook de práctica de esta clase.** Vas a completar tú misma o tú mismo la pérdida de DQN desde cero y vas a programar un decaimiento de epsilon para compararlo con el epsilon fijo que usaste hoy.

DQN aprende valores de acción y deriva la política de ellos, de forma indirecta, escogiendo siempre la acción con mayor Q. En la clase ocho vas a conocer otra familia de métodos, los gradientes de política y los métodos actor-crítico, que aprenden la política de forma directa, sin pasar por una función de valor intermedia.
